# GWAS-PPI 解析パイプライン (pj02)

## 概要
疾患はゲノム構造の影響を受けた下流の遺伝子が引き起こす生体メカニズムの異常と定義する。
上流（ゲノム）から下流（疾患メカニズム）の間で中心を担う遺伝子を見つけることで創薬標的探索を見出す。

### パイプライン
1. **GWAS SNP → 遺伝子マッピング**: 特定疾患のSNPsを調査し、SNP関連遺伝子を特定
2. **PPI ネットワーク構築**: SNP関連遺伝子と繋がりのある遺伝子群を取得 (1～3階層)
3. **フローベース中心性解析**: GWAS P値 + variant影響度をフローとし、重要遺伝子をスコア化
4. **エンリッチメント解析**: Reactome, GO, HPO によるローカル解析
5. **可視化**: サンキーダイアグラム、ネットワーク図、パスウェイ-遺伝子マトリックス

---
## 0. セットアップ

In [ ]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import config
import gwas_fetcher
import ppi_fetcher
import gene_scorer
import network_analysis
import enrichment
import visualization
import id_mapper

import pandas as pd
import numpy as np
import networkx as nx
import plotly.io as pio

pio.renderers.default = 'notebook'

print('✅ 全モジュール読み込み完了')
print(f'出力ディレクトリ: {config.OUTPUT_DIR}')
import target_fetcher


---
## 1. パラメータ設定
ここで解析対象の疾患名やパラメータを設定します。

In [ ]:
# ========================================
# ★ ここを変更して解析対象を設定 ★
# ========================================

# 疾患名 (GWAS Catalog で検索)
DISEASE_TRAIT = "Type 2 Diabetes"

# EFO IDで検索する場合 (Noneならdisease_traitで検索)
EFO_TRAIT_ID = None  # 例: "EFO_0001360"

# PPI ソース選択 (デフォルト: SIGNOR, Reactome, STRING(Score>0.7))
PPI_SOURCES = ["signor", "reactome", "string"]  # "biogrid" も追加可能

# PPI 階層数 (1～3, デフォルト: 1)
PPI_LAYERS = 1

# STRING スコア閾値 (700 = 0.7)
STRING_MIN_SCORE = 700

# エンリッチメントに使用するデータベース
ENRICHMENT_DBS = ["reactome", "go", "hpo"]

# FDR 閾値
FDR_THRESHOLD = 0.05

# 重要遺伝子の最大数
TOP_N_KEY_GENES = 50

print(f'疾患: {DISEASE_TRAIT}')
print(f'PPI ソース: {PPI_SOURCES}')
print(f'PPI 階層: {PPI_LAYERS}')
print(f'STRING閾値: {STRING_MIN_SCORE/1000:.1f}')
print(f'Enrichment DB: {ENRICHMENT_DBS}')

---
## 2. GWAS SNP → 遺伝子マッピング
特定の疾患のSNPsを調査し、SNPに関連のある遺伝子（SNP関連遺伝子）を特定する。

In [ ]:
# GWAS データ取得
if EFO_TRAIT_ID:
    gwas_df = gwas_fetcher.fetch_snps_by_efo_trait(EFO_TRAIT_ID)
else:
    gwas_df = gwas_fetcher.fetch_snps_for_disease(DISEASE_TRAIT)

print(f'\n取得結果:')
print(f'  SNPs: {gwas_df["rsid"].nunique()}')
print(f'  遺伝子: {gwas_df["gene_symbol"].nunique()}')
print(f'  アソシエーション: {len(gwas_df)}')

gwas_genes = gwas_df['gene_symbol'].unique().tolist()
print(f'\nSNP関連遺伝子リスト ({len(gwas_genes)}): {gwas_genes[:20]}...')

gwas_df.head(10)

---
## 3. PPI ネットワーク構築
SNP関連遺伝子と繋がりのある遺伝子群（PPI遺伝子群）を調べる。
- Sources: SIGNOR, Reactome, STRING (Score>0.7)
- 階層: 設定値 (デフォルト: 1階層)

In [ ]:
# 多階層 PPI 取得
ppi_df = ppi_fetcher.fetch_multi_layer_ppi(
    seed_genes=gwas_genes,
    layers=PPI_LAYERS,
    sources=PPI_SOURCES,
    string_min_score=STRING_MIN_SCORE,
)

if not ppi_df.empty:
    ppi_genes = set(ppi_df['gene_a']) | set(ppi_df['gene_b'])
    ppi_only_genes = ppi_genes - set(gwas_genes)
    print(f'\nPPI結果:')
    print(f'  インタラクション数: {len(ppi_df)}')
    print(f'  ネットワーク内遺伝子: {len(ppi_genes)}')
    print(f'  PPI固有遺伝子: {len(ppi_only_genes)}')
    
    # ソース別統計
    print(f'\nソース別:')
    for source, group in ppi_df.groupby('source'):
        print(f'  {source}: {len(group)} インタラクション')
    
    ppi_df.head(10)
else:
    print('PPI データなし')

---
## 4. 遺伝子スコアリング
SNP関連遺伝子のフロー（初期流量）を定義:
- GWAS P値 (-log10)
- LoF, GoF, Missense 等の variant 影響度

In [ ]:
# バリアント機能アノテーション取得 (Ensembl VEP)
rsids = gwas_df['rsid'].unique().tolist()
print(f'{len(rsids)} バリアントのアノテーション取得中...')

vep_df = gene_scorer.fetch_variant_consequences(rsids)
variant_classes = pd.DataFrame()

if not vep_df.empty:
    variant_classes = gene_scorer.classify_variant_effects(vep_df)
    print(f'\nバリアント分類結果:')
    if not variant_classes.empty:
        print(f'  LoF: {variant_classes["is_lof"].sum()} 遺伝子')
        print(f'  GoF: {variant_classes["is_gof"].sum()} 遺伝子')
        print(f'  Missense: {variant_classes["is_missense"].sum()} 遺伝子')
        print(f'  Coding: {variant_classes["is_coding"].sum()} 遺伝子')
        print(f'  Regulatory: {variant_classes["is_regulatory"].sum()} 遺伝子')
else:
    print('VEP データなし（スコアリングはGWAS P値のみで実施）')

In [ ]:
# 遺伝子スコアリング
gene_scores_df = gene_scorer.score_genes(
    gwas_df=gwas_df,
    variant_classifications=variant_classes,
    ppi_df=ppi_df,
)

print(f'\n遺伝子スコア (Top 15):')
gene_scores_df.head(15)

In [ ]:
# 遺伝子スコア可視化
fig_scores = visualization.create_gene_score_plot(
    gene_scores_df, top_n=30,
    save_path=os.path.join(config.OUTPUT_DIR, 'gene_scores.html')
)
fig_scores.show()

---
## 5. フローベース中心性解析
SNP関連遺伝子からのフローをベースにネットワークの流れを計算し、重要遺伝子をスコア化する。

In [ ]:
# ネットワーク構築
gene_score_dict = dict(zip(gene_scores_df['gene_symbol'], gene_scores_df['total_score']))

G = network_analysis.build_ppi_network(
    ppi_df=ppi_df,
    gwas_genes=gwas_genes,
    gene_scores=gene_score_dict,
)

# 中心性指標計算
centrality_df = network_analysis.compute_centrality_metrics(G)

# フロー伝播
flow_scores = network_analysis.propagate_flow_scores(
    G=G,
    gene_scores_df=gene_scores_df,
    damping=0.5,
    iterations=5,
)

# 統合スコア計算
integrated_df = network_analysis.compute_integrated_scores(
    G=G,
    centrality_df=centrality_df,
    flow_scores=flow_scores,
)

print(f'\n統合スコア (Top 20):')
integrated_df.head(20)

In [ ]:
# 重要遺伝子選択
key_genes_df = network_analysis.select_key_genes(
    integrated_df,
    top_n=TOP_N_KEY_GENES,
    min_score_percentile=75,
    include_all_gwas=True,
)

key_gene_list = key_genes_df['gene_symbol'].tolist()
print(f'\n重要遺伝子 ({len(key_gene_list)}):')
print(key_gene_list)

key_genes_df.head(20)

---
## 6. エンリッチメント解析
重要遺伝子群を用いたエンリッチメント解析をローカルで実施:
- Reactome
- Gene Ontology (GO)
- HPO (Human Phenotype Ontology)

In [ ]:
# 全 Enrichment 解析実行
enrichment_df = enrichment.run_all_enrichment(
    gene_list=key_gene_list,
    databases=ENRICHMENT_DBS,
    fdr_threshold=FDR_THRESHOLD,
)

if not enrichment_df.empty:
    # 結果保存
    enrichment_df.to_csv(
        os.path.join(config.OUTPUT_DIR, 'enrichment_results.csv'),
        index=False,
    )
    print(f'\n結果を保存: {config.OUTPUT_DIR}/enrichment_results.csv')
    enrichment_df.head(20)
else:
    print('有意なエンリッチメント結果なし')

In [ ]:
# エンリッチメント可視化: データベースごとにバブルチャート
if not enrichment_df.empty:
    for db_name in enrichment_df['database'].unique():
        db_df = enrichment_df[enrichment_df['database'] == db_name].copy()
        if not db_df.empty:
            print(f'\n=== {db_name} Enrichment: {len(db_df)} 有意ターム ===')
            fig_bubble = visualization.create_enrichment_bubble_chart(
                db_df, top_n=20,
                save_path=os.path.join(config.OUTPUT_DIR, f'enrichment_bubble_{db_name.lower()}.html'),
            )
            fig_bubble.update_layout(title=f'{db_name} Enrichment (Top 20)')
            fig_bubble.show()

In [ ]:
# エンリッチメント可視化: データベースごとに棒グラフ
if not enrichment_df.empty:
    for db_name in enrichment_df['database'].unique():
        db_df = enrichment_df[enrichment_df['database'] == db_name].copy()
        if not db_df.empty:
            fig_bar = visualization.create_enrichment_barplot(
                db_df, top_n=20,
                save_path=os.path.join(config.OUTPUT_DIR, f'enrichment_bar_{db_name.lower()}.html'),
            )
            fig_bar.update_layout(title=f'{db_name} Enrichment (-log10 FDR, Top 20)')
            fig_bar.show()

---
## 7. 可視化
- サンキーダイアグラム: SNP → SNP関連遺伝子 → PPI重要遺伝子
- ネットワーク図 (Plotly)

In [ ]:
# サンキーダイアグラム
fig_sankey = visualization.create_sankey_diagram(
    gwas_df=gwas_df,
    ppi_df=ppi_df,
    key_genes_df=key_genes_df,
    enrichment_df=enrichment_df if not enrichment_df.empty else None,
    top_n_pathways=15,
    save_path=os.path.join(config.OUTPUT_DIR, 'sankey_flow.html'),
)
fig_sankey.show()

In [ ]:
# PPI ネットワーク図 (Plotly)
fig_network = visualization.create_network_plot(
    G=G,
    key_genes=key_gene_list,
    gene_scores=flow_scores,
        known_targets=known_targets,
    save_path=os.path.join(config.OUTPUT_DIR, 'ppi_network_interactive.html'),
)
fig_network.show()

---
## 8. パスウェイ-遺伝子マトリックス
エンリッチメント解析の重要パスウェイごとに重要遺伝子をリストアップして示す。

In [ ]:
# パスウェイ-遺伝子マトリックス
if not enrichment_df.empty:
    fig_matrix = visualization.create_pathway_gene_matrix(
        enrichment_df=enrichment_df,
        key_genes=key_gene_list,
    gene_scores=flow_scores,
        known_targets=known_targets,
        top_n_pathways=15,
        save_path=os.path.join(config.OUTPUT_DIR, 'pathway_gene_matrix.html'),
    )


In [ ]:
# パスウェイごとの重要遺伝子リスト (テーブル)
if not enrichment_df.empty:
    print('\n=== パスウェイ別 重要遺伝子リスト ===')
    print('='*80)
    
    key_set = set(g.upper() for g in key_gene_list)
    
    for i, row in enrichment_df.head(20).iterrows():
        pw_genes = set(g.strip().upper() for g in row['genes'].split(','))
        key_in_pw = sorted(pw_genes & key_set)
        
        if key_in_pw:
            print(f'\n[{row["database"]}] {row["term_name"]}')
            print(f'  FDR: {row["fdr"]:.2e} | Fold: {row["fold_enrichment"]:.1f}x')
            print(f'  重要遺伝子 ({len(key_in_pw)}): {", ".join(key_in_pw)}')

---
## 9. データ保存

In [ ]:
# 結果テーブルの保存
output_dir = config.OUTPUT_DIR

# GWAS SNP データ
gwas_df.to_csv(os.path.join(output_dir, 'gwas_snps.csv'), index=False)

# PPI インタラクション
if not ppi_df.empty:
    ppi_df.to_csv(os.path.join(output_dir, 'ppi_interactions.csv'), index=False)

# 遺伝子スコア
gene_scores_df.to_csv(os.path.join(output_dir, 'gene_scores.csv'), index=False)

# 統合スコア
if not integrated_df.empty:
    integrated_df.to_csv(os.path.join(output_dir, 'integrated_scores.csv'), index=False)

# 重要遺伝子
key_genes_df.to_csv(os.path.join(output_dir, 'key_genes.csv'), index=False)

print(f'✅ 全データを {output_dir} に保存完了')
print(f'\n保存ファイル一覧:')
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f'  {f} ({size:,} bytes)')

## 10. パスウェイ別ネットワーク可視化
各エンリッチメントパスウェイに含まれる遺伝子だけを抽出し、PPIサブネットワークを描画します。

In [ ]:
top_pw = enrichment_df.head(5)  # 上位5つのパスウェイを描画

for i, row in top_pw.iterrows():
    pw_name = row['term_name']
    pw_genes = [g.strip().upper() for g in row['genes'].split(',')]
    
    safe_name = "".join([c if c.isalnum() else "_" for c in pw_name[:30]])
    save_path = os.path.join(config.OUTPUT_DIR, f'pathway_net_{i+1}_{safe_name}.html')
    
    fig_pw = visualization.create_pathway_network_plot(
        G=ppi_graph,
        pathway_name=pw_name,
        pathway_genes=pw_genes,
        gene_scores=flow_scores,
        known_targets=known_targets,
        save_path=save_path
    )
    if len(fig_pw.data) > 0:
        fig_pw.show()